In [66]:
import os, dotenv
dotenv.load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY", "not_set")
api_key = os.getenv("TOGETHER_API_KEY", api_key)


In [69]:
from openai import OpenAI

# load API key from env


client = OpenAI(
    api_key=api_key,
   #  base_url="https://api.together.xyz/v1",
   base_url = "https://openrouter.ai/api/v1"
)


MODEL = "openai/gpt-oss-120B"  # or leave blank for default
# Make the shared prefix very long and ensure it's the very first tokens
SYSTEM_BLOCK = (
   "You are a concise JSON-only assistant. "
   "Respond with a compact JSON object. "
   "If additional detail exists, include \"expandable\": true. \n"
)
POLICY_LINE = (
   "Rule: Do not add commentary. Prefer short keys. "
   "Use stable field order. Avoid repeating unchanged values.\n"
)
SYSTEM_BLOCK = SYSTEM_BLOCK + (POLICY_LINE * 400)
# Put all reusable context into the *system* message (earliest position).
CONTEXT = (
   "Policy A: Users may request summaries; keep responses under 80 tokens by default.\n\n"
   "Policy B: For lists, cap items at 5 unless user asks to expand.\n\n"
   "Policy C: Use ISO dates; currencies in USD.\n"
)
print("System block token count:", len(SYSTEM_BLOCK.split()))

System block token count: 6419


In [64]:
from sympy.physics.quantum.sho1d import m

def make_messages(user_question: str):
   return [
       {"role": "system", "content": SYSTEM_BLOCK + "\n" + CONTEXT},
       {"role": "user",   "content": user_question}
   ]
def call_once(q):
   r = client.chat.completions.create(
       model=MODEL,
       messages=make_messages(q),
       temperature=0,
       max_tokens=200,
    extra_body={"reasoning": {"effort": "low"}, "provider": {"order": ["deepinfra"], "allow_fallbacks": False}, "prompt_cache_key": "user848-chat333"}
   )
   u = getattr(r, "usage", None)
   ptd = getattr(u, "prompt_tokens_details", None)
   print("\n=== API Response ===")
   print(u)
   print(ptd)
   print(r)





print("Cold:")  # expect low cache
call_once("Summarize the policies in 3 bullets.")
print("Warm:")  # expect very high cache
call_once("List 3 risks if these rules are ignored.")  # identical tail; expect very high cache

Cold:

=== API Response ===
CompletionUsage(completion_tokens=143, prompt_tokens=8557, total_tokens=8700, completion_tokens_details=None, prompt_tokens_details=None, reasoning_tokens=0)
None
ChatCompletion(id='ofmC9WC-z1gNr-9eebe4074d2e8b1c', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"bullets":["Keep summaries ≤80 tokens by default.","List items capped at 5 unless expanded request.","Use ISO dates and USD for currencies."],"expandable":true}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='We need to output JSON only, compact, with possibly "expandable": true if more detail exists. Summarize policies in 3 bullets. So JSON object with maybe "bullets": [list of 3 strings], and "expandable": true because more detail exists. Follow stable field order: maybe "bullets" then "expandable". No extra commentary. Ensure short keys. Provide 3 bullet strings summarizing policies 

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_harmony")

In [74]:
import tiktoken

text = "Hello, this is a test."
n_tokens = len(enc.encode(text, allowed_special="all"))

print(n_tokens)

7


In [70]:

response = client.responses.input_tokens.count(
    model=MODEL,
    input="Tell me a joke."
)
print(response.input_tokens)

NotFoundError: Error code: 404 - {'error': {'message': 'Not Found', 'code': 404}}